## Fetching movie data

In this Jupyter notebook, I will fetch the movie details. Since my target audience is in Mexico, I will retrieve the data in Spanish. Then, I will merge this information with the final movie details CSV, which is in English, in order to add additional data such as the director and a personalized score.

In [23]:
import pandas as pd
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import warnings
warnings.filterwarnings('ignore')

import os
from dotenv import load_dotenv
# Load the environment variables .env
load_dotenv()

True

In [24]:
mdf = pd.read_csv('../data/processed/movies_final.csv')
mdf.head()

,old_id,id,title,genres,overview,runtime,release_date,tagline,vote_count,vote_average,poster_path,backdrop_path,cast,director,keywords,score,movieId
0,221850,614696,#Alive,"['Action', 'Horror', 'Science Fiction']","As a grisly virus rampages a city, a lone man ...",98,2020-06-24,You must survive.,1927,7.222,/lZPvLUMYEPLTE2df1VW5FHTYC8N.jpg,/k2SY15W9QXH9qL8f4a4BbytV1BE.jpg,"['Yoo Ah-in', 'Park Shin-hye', 'Lee Hyun-wook']",Cho Il,"['escape', 'alone', 'survival', 'drone', 'zomb...",7.220984,0
1,117867,252178,'71,"['Thriller', 'Action', 'Drama', 'War']",A young British soldier must find his way back...,99,2014-10-10,NaN,1155,6.802,/wbhqBocsP7QoX8SZLvCsGOWUAaQ.jpg,/aTloiKdNs2c8vlstbx3wBWD6Thi.jpg,"[""Jack O'Connell"", 'Sean Harris', 'Paul Anders...",Yann Demange,"['1970s', 'riot', 'northern ireland', 'surviva...",6.822391,1
2,69757,19913,(500) Days of Summer,"['Comedy', 'Drama', 'Romance']","Tom, greeting-card writer and hopeless romanti...",95,2009-07-17,This is not a love story. This is a story abou...,10548,7.296,/qXAuQ9hF30sQRsXf40OfRVl0MJZ.jpg,/1M2i4Mxd03elGOTmEkIvqrHfmyS.jpg,"['Joseph Gordon-Levitt', 'Zooey Deschanel', 'G...",Marc Webb,"['jealousy', 'gallery', 'fight', 'date', 'arch...",7.295363,2
3,152077,333371,10 Cloverfield Lane,"['Thriller', 'Science Fiction', 'Drama', 'Horr...","After a catastrophic car crash, a young woman ...",104,2016-03-10,Monsters come in many forms.,8238,6.992,/84Dhwz93vCin6T1PX6ctSvWEuNE.jpg,/veGaHYcRHFPEoKfqxKbCEXI8tOT.jpg,"['John Goodman', 'Mary Elizabeth Winstead', 'J...",Dan Trachtenberg,"['kidnapping', 'paranoia', 'bunker', 'basement...",6.993529,3
4,2572,4951,10 Things I Hate About You,"['Comedy', 'Romance', 'Drama']","On the first day at his new school, Cameron in...",97,1999-03-30,How do I loathe thee? Let me count the ways.,8498,7.596,/ujERk3aKABXU3NDXOAxEQYTHe9A.jpg,/yvPbncYhMu9FfTjDhq0N5lgnVkO.jpg,"['Heath Ledger', 'Julia Stiles', 'Joseph Gordo...",Gil Junger,"['high school', 'deception', 'based on play or...",7.592968,4


In [25]:
def fetch_movie(movie_id, api_key):
    """
    Function to fetch data for a single movie from TMDB API
    Args:
        movie_id (int): TMDB_ID of the movie
        api_key (str): API key for authentication

    Returns:
        dict: Movie data if request is successful, None if failed
    """
    url = f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={api_key}&language=es-Mx"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            return response.json()
        else:
            return None  # Return None if response code is not 200 (success)

    except requests.exceptions.RequestException as e:
        # Handle any network-related or request errors
        print(f"Error fetching data for movie ID {movie_id}: {e}")
        return None

In [26]:
def fetch_movies(ids, api_key):
    """
    Function to fetch movie data for multiple movie IDs using concurrent requests
    Args:
        ids (list): List of movie IDs to fetch
        api_key (str): API key for authentication

    Returns:
        tuple: A tuple containing two lists:
            - List of successfully fetched movies (as JSON)
            - List of movie IDs for which fetching data failed
    """
    id_errors = []
    movies = []

    # Using ThreadPoolExecutor to send multiple requests concurrently
    with ThreadPoolExecutor(max_workers=10) as executor:
        # Submitting the fetch_movie function to the executor for each movie ID
        futures = {executor.submit(fetch_movie, movie_id, api_key): movie_id for movie_id in ids}

        # Processing results as they complete
        for future in as_completed(futures):
            movie_id = futures[future]
            movie_data = future.result()

            if movie_data:
                movies.append(movie_data)
            else:
                id_errors.append(movie_id)

            # Adding a small sleep time to avoid hitting API rate limits
            time.sleep(0.005)

    return movies, id_errors

In [27]:
ids = mdf['id'].to_list()
ids[:5]

[614696, 252178, 19913, 333371, 4951]

In [28]:
api_key = os.getenv('tmdb_api_key')
movies, id_errors = fetch_movies(ids, api_key)

print(f"Fetched {len(movies)} movies")
print(f"Failed to fetch {len(id_errors)} movies")

Fetched 4946 movies
Failed to fetch 0 movies


In [29]:
fetched = pd.DataFrame(data=movies)
fetched.head().transpose()

,0,1,2,3,4
adult,False,False,False,False,False
backdrop_path,/aTloiKdNs2c8vlstbx3wBWD6Thi.jpg,/k2SY15W9QXH9qL8f4a4BbytV1BE.jpg,/sWjfx6KuENXi76yVc4bkl18jtBI.jpg,/4ssWRanWTKN9CQ0tfq5S1whP7tr.jpg,/1M2i4Mxd03elGOTmEkIvqrHfmyS.jpg
belongs_to_collection,None,None,None,"{'id': 1477103, 'name': '10 Things I Hate Abou...",None
budget,11000000,6300000,15000000,16000000,7500000
genres,"[{'id': 53, 'name': 'Suspense'}, {'id': 28, 'n...","[{'id': 28, 'name': 'Acción'}, {'id': 27, 'nam...","[{'id': 53, 'name': 'Suspense'}, {'id': 878, '...","[{'id': 35, 'name': 'Comedia'}, {'id': 10749, ...","[{'id': 35, 'name': 'Comedia'}, {'id': 18, 'na..."
homepage,,,,,
id,252178,614696,333371,4951,19913
imdb_id,tt2614684,tt10620868,tt1179933,tt0147800,tt1022603
origin_country,[GB],[KR],[US],[US],[US]
original_language,en,ko,en,en,en


In [30]:
# Select only the relevant columns
fetched = fetched[['id', 'title', 'genres', 'overview', 
           'release_date', 'runtime', 'tagline',  'vote_average', 
           'popularity', 'vote_count', 'poster_path', 'backdrop_path']]

fetched.head().transpose()

,0,1,2,3,4
id,252178,614696,333371,4951,19913
title,'71,#Vivo,Avenida Cloverfield 10,10 cosas que odio de ti,(500) días con ella
genres,"[{'id': 53, 'name': 'Suspense'}, {'id': 28, 'n...","[{'id': 28, 'name': 'Acción'}, {'id': 27, 'nam...","[{'id': 53, 'name': 'Suspense'}, {'id': 878, '...","[{'id': 35, 'name': 'Comedia'}, {'id': 10749, ...","[{'id': 35, 'name': 'Comedia'}, {'id': 18, 'na..."
overview,"Corre el año 1971, y la tensión social en Irla...",La rápida propagación de una infección descono...,Una joven sufre un accidente de coche. Cuando ...,Las hermanas Stratford son muy distintas. La b...,"Tom aún sigue creyendo, incluso en este cínico..."
release_date,2014-10-10,2020-06-24,2016-03-10,1999-03-30,2009-07-17
runtime,99,98,103,97,95
tagline,,,Los monstruos vienen en muchas formas.,¿Por qué te detesto? Déjame Contarte,"Él encontró el amor de su vida, ella no."
vote_average,6.805,7.218,7.0,7.59,7.3
popularity,1.9949,8.3511,8.0165,13.6231,10.1919
vote_count,1175,1971,8411,8675,10748


In [31]:
mdf.columns

Index(['old_id', 'id', 'title', 'genres', 'overview', 'runtime',
       'release_date', 'tagline', 'vote_count', 'vote_average', 'poster_path',
       'backdrop_path', 'cast', 'director', 'keywords', 'score', 'movieId'],
      dtype='object')

In [32]:
# Select the missing columns
mdf = mdf[['id', 'old_id', 'movieId', 'cast', 'director', 'keywords', 'score']]
mdf.head()

,id,old_id,movieId,cast,director,keywords,score
0,614696,221850,0,"['Yoo Ah-in', 'Park Shin-hye', 'Lee Hyun-wook']",Cho Il,"['escape', 'alone', 'survival', 'drone', 'zomb...",7.220984
1,252178,117867,1,"[""Jack O'Connell"", 'Sean Harris', 'Paul Anders...",Yann Demange,"['1970s', 'riot', 'northern ireland', 'surviva...",6.822391
2,19913,69757,2,"['Joseph Gordon-Levitt', 'Zooey Deschanel', 'G...",Marc Webb,"['jealousy', 'gallery', 'fight', 'date', 'arch...",7.295363
3,333371,152077,3,"['John Goodman', 'Mary Elizabeth Winstead', 'J...",Dan Trachtenberg,"['kidnapping', 'paranoia', 'bunker', 'basement...",6.993529
4,4951,2572,4,"['Heath Ledger', 'Julia Stiles', 'Joseph Gordo...",Gil Junger,"['high school', 'deception', 'based on play or...",7.592968


In [33]:
spanish_df = pd.merge(fetched, mdf, on='id')
spanish_df.head().transpose()

,0,1,2,3,4
id,252178,614696,333371,4951,19913
title,'71,#Vivo,Avenida Cloverfield 10,10 cosas que odio de ti,(500) días con ella
genres,"[{'id': 53, 'name': 'Suspense'}, {'id': 28, 'n...","[{'id': 28, 'name': 'Acción'}, {'id': 27, 'nam...","[{'id': 53, 'name': 'Suspense'}, {'id': 878, '...","[{'id': 35, 'name': 'Comedia'}, {'id': 10749, ...","[{'id': 35, 'name': 'Comedia'}, {'id': 18, 'na..."
overview,"Corre el año 1971, y la tensión social en Irla...",La rápida propagación de una infección descono...,Una joven sufre un accidente de coche. Cuando ...,Las hermanas Stratford son muy distintas. La b...,"Tom aún sigue creyendo, incluso en este cínico..."
release_date,2014-10-10,2020-06-24,2016-03-10,1999-03-30,2009-07-17
runtime,99,98,103,97,95
tagline,,,Los monstruos vienen en muchas formas.,¿Por qué te detesto? Déjame Contarte,"Él encontró el amor de su vida, ella no."
vote_average,6.805,7.218,7.0,7.59,7.3
popularity,1.9949,8.3511,8.0165,13.6231,10.1919
vote_count,1175,1971,8411,8675,10748


In [34]:
# Fix the columns names 
spanish_df = spanish_df.rename(columns={'id': 'TMDB_id', 'movieId': 'id', 'old_id' : 'MovieLens_id'})
# Sort the columns by relevance
final_df = spanish_df[['MovieLens_id', 'id', 'TMDB_id', 'title', 'genres', 
       'overview', 'score', 'release_date', 'runtime', 'tagline', 
       'vote_average', 'vote_count', 'popularity', 'cast', 'director', 
       'keywords', 'poster_path', 'backdrop_path' 
       ]]
# Sort the movies by internal id
final_df.sort_values(by='id', ascending=True, inplace=True)
# Extract the genres from the json format
final_df['genres'] = final_df['genres'].apply(lambda x: [item['name'] for item in x])

final_df.head()

,MovieLens_id,id,TMDB_id,title,genres,overview,score,release_date,runtime,tagline,vote_average,vote_count,popularity,cast,director,keywords,poster_path,backdrop_path
1,221850,0,614696,#Vivo,"[Acción, Terror, Ciencia ficción]",La rápida propagación de una infección descono...,7.220984,2020-06-24,98,,7.218,1971,8.3511,"['Yoo Ah-in', 'Park Shin-hye', 'Lee Hyun-wook']",Cho Il,"['escape', 'alone', 'survival', 'drone', 'zomb...",/tmGOSFUjCjWJ6ASxfCbCjEyamx.jpg,/k2SY15W9QXH9qL8f4a4BbytV1BE.jpg
0,117867,1,252178,'71,"[Suspense, Acción, Drama, Bélica]","Corre el año 1971, y la tensión social en Irla...",6.822391,2014-10-10,99,,6.805,1175,1.9949,"[""Jack O'Connell"", 'Sean Harris', 'Paul Anders...",Yann Demange,"['1970s', 'riot', 'northern ireland', 'surviva...",/mdychWI794HVjCjeJjAk3FdJgeq.jpg,/aTloiKdNs2c8vlstbx3wBWD6Thi.jpg
4,69757,2,19913,(500) días con ella,"[Comedia, Drama, Romance]","Tom aún sigue creyendo, incluso en este cínico...",7.295363,2009-07-17,95,"Él encontró el amor de su vida, ella no.",7.300,10748,10.1919,"['Joseph Gordon-Levitt', 'Zooey Deschanel', 'G...",Marc Webb,"['jealousy', 'gallery', 'fight', 'date', 'arch...",/xziurNdvxWZkgonZ5ZaGRB9YdLt.jpg,/1M2i4Mxd03elGOTmEkIvqrHfmyS.jpg
2,152077,3,333371,Avenida Cloverfield 10,"[Suspense, Ciencia ficción, Drama, Terror]",Una joven sufre un accidente de coche. Cuando ...,6.993529,2016-03-10,103,Los monstruos vienen en muchas formas.,7.000,8411,8.0165,"['John Goodman', 'Mary Elizabeth Winstead', 'J...",Dan Trachtenberg,"['kidnapping', 'paranoia', 'bunker', 'basement...",/zDdmGyJ2uTYscOz1bVslQTPrp1a.jpg,/sWjfx6KuENXi76yVc4bkl18jtBI.jpg
3,2572,4,4951,10 cosas que odio de ti,"[Comedia, Romance, Drama]",Las hermanas Stratford son muy distintas. La b...,7.592968,1999-03-30,97,¿Por qué te detesto? Déjame Contarte,7.590,8675,13.6231,"['Heath Ledger', 'Julia Stiles', 'Joseph Gordo...",Gil Junger,"['high school', 'deception', 'based on play or...",/ckKmeIh1yo4AMvNUsa2E6NGpBgj.jpg,/4ssWRanWTKN9CQ0tfq5S1whP7tr.jpg


In [35]:
# Check if there is an important null field
final_df[['title', 'id', 'poster_path', 'backdrop_path']].isna().sum()

title            0
id               0
poster_path      0
backdrop_path    0
dtype: int64

In [36]:
final_df.to_csv('../data/processed/movies_final_spanish.csv', index=False)